# webGrainy — Genuinely Held-Out Follow-Up Validation & Ablation (Workstream A)

This self-contained notebook addresses **AUDIT_REPORT.md F-02** and **Workstream A** by:
1. **Fixing the `split`/`site` bug** from `imagegrains_full_dataset_validation_colab_v2.ipynb` (extracts to `/content/data/IG2_data` instead of `/content/test_images/` and parses `<SITE>/<train|test>/<file>`).
2. **Evaluating on the true `official_split == 'test'` set (`n = 40`)** across all 13 sites in `IG2_data.zip` (with an optional `EVAL_SCOPE` toggle for `all_243` or leave-one-site-out checks).
3. **Executing the actual shipped TypeScript engine** (`src/modules/postprocessCore.ts` + `src/modules/morphometricsExt.ts` via `audit/followup/ts_postprocess_runner.mjs` under Node.js) with frozen v1.1.0 defaults (`niter=250, cellprobThreshold=0.0, flowThreshold=0.45, minSize=30, minSeedDensity=2, excludeBorderMargin=2`), rather than a Python re-implementation.
4. **Ablating the `0.15` upper-area guard** (`maxGrainAreaFraction = 0.15` vs `1.0`) for both **U-Net** and **Cellpose-SAM**, alongside the **upstream PyTorch ImageGrains** reference.
5. **Verifying `onnxruntime-web` WASM parity** against Python `onnxruntime` on 3 test images.
6. **Reporting D16 / D50 / D84 errors with bootstrap 95% CIs, Panoptic Quality (PQ), and `bAxis` / `dEq` / `minFeret` D50 shifts** (for Workstream C), plus comparing `demo_data/K1/K1_C2_385.jpg` & `K1_C3_0449.jpg` (`0.39 mm/px`) against `demo_data/K1_field_measurement.csv` (`n = 39`, noting K1's overlap with `APF_2`).

Outputs saved to `/content/followup_outputs/heldout_results_per_image.csv` and `/content/followup_outputs/heldout_summary.json` (plus printed JSON at the end for easy copy-paste).

In [ ]:
# Cell 1: Setup repo, Node.js dependencies, Python packages, and isolated Cellpose 3 & 4 venvs
import os, subprocess, sys

REPO_DIR = "/content/webGrainy"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/LukeAFullard/webGrainy.git", REPO_DIR], check=True)

subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
chk = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--verify", "origin/audit/followup-v1.1.0"], capture_output=True)
target_ref = "origin/audit/followup-v1.1.0" if chk.returncode == 0 else "origin/main"
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", "followup_run", target_ref], check=True)
print("Checked out", target_ref, "->", subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip())

# Install Node dependencies (esbuild + onnxruntime-web)
subprocess.run(["npm", "ci", "--no-audit", "--no-fund"], cwd=REPO_DIR, check=True)

# Verify ts_postprocess_runner.mjs self-test
subprocess.run(["node", "audit/followup/ts_postprocess_runner.mjs", "--selftest"], cwd=REPO_DIR, check=True)

# Install Python scientific stack in main runtime
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "uv", "onnxruntime-gpu", "scikit-image", "scipy", "pandas", "opencv-python-headless", "tqdm", "pillow"
], check=True)

def verify_venv(py_bin: str, label: str) -> bool:
    if not os.path.exists(py_bin):
        return False
    res = subprocess.run(
        [py_bin, "-c", "import numpy, torch, cv2; from cellpose import models"],
        capture_output=True, text=True
    )
    if res.returncode != 0:
        print(f"[{label}] Rebuild needed: {res.stderr.strip().splitlines()[-1] if res.stderr.strip() else res.returncode}")
        return False
    return True

# Isolated Cellpose 3 (U-Net IG2_full_set.200525) & Cellpose 4 (CP-SAM IG2_full_set_cp_SAM)
if not verify_venv("/content/venv_cp3/bin/python", "venv_cp3"):
    print("Building /content/venv_cp3 (cellpose==3.1.1.1)...")
    subprocess.run(["uv", "venv", "/content/venv_cp3", "--python", "3.11", "--clear", "-q"], check=True)
    subprocess.run([
        "uv", "pip", "install", "--python", "/content/venv_cp3/bin/python", "-q",
        "cellpose==3.1.1.1", "torch", "opencv-python-headless", "numpy", "pillow"
    ], check=True)
    assert verify_venv("/content/venv_cp3/bin/python", "venv_cp3"), "venv_cp3 failed verification"

if not verify_venv("/content/venv_cp4/bin/python", "venv_cp4"):
    print("Building /content/venv_cp4 (cellpose>=4.0.9,<4.2)...")
    subprocess.run(["uv", "venv", "/content/venv_cp4", "--python", "3.11", "--clear", "-q"], check=True)
    subprocess.run([
        "uv", "pip", "install", "--python", "/content/venv_cp4/bin/python", "-q",
        "cellpose>=4.0.9,<4.2", "torch", "opencv-python-headless", "numpy", "pillow"
    ], check=True)
    assert verify_venv("/content/venv_cp4/bin/python", "venv_cp4"), "venv_cp4 failed verification"

import torch
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Both venv_cp3 and venv_cp4 ready.")

In [ ]:
# Cell 2: Mount Drive (if in Colab), reuse cached IG2_data.zip/weights, or download & verify SHA-256
import csv, hashlib, os, shutil, urllib.request, zipfile
from collections import Counter
from tqdm.auto import tqdm

DRIVE_ROOT = "/content/drive/MyDrive/imagegrains_full_validation"
DRIVE_DATA_DIR = "/content/drive/MyDrive/imagegrains_datasets"
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    DRIVE_MOUNTED = os.path.exists("/content/drive/MyDrive")
except Exception as e:
    print("Drive mount skipped:", e)

DATA_DIR = "/content/data/IG2_data"
WEIGHTS_DIR = "/content/weights"
K1_DIR = "/content/data/K1_demo"
OUT_DIR = "/content/followup_outputs"
for d in [DATA_DIR, WEIGHTS_DIR, K1_DIR, OUT_DIR]:
    os.makedirs(d, exist_ok=True)

def find_cached_on_drive(filename: str) -> str | None:
    if not DRIVE_MOUNTED:
        return None
    for base in [
        DRIVE_DATA_DIR,
        os.path.join(DRIVE_DATA_DIR, "models"),
        os.path.join(DRIVE_DATA_DIR, "weights"),
        DRIVE_ROOT,
        os.path.join(DRIVE_ROOT, "models"),
        os.path.join(DRIVE_ROOT, "weights"),
    ]:
        cand = os.path.join(base, filename)
        if os.path.exists(cand) and os.path.getsize(cand) > 0:
            return cand
    return None

def download_file(url: str, dst: str, force: bool = False):
    if not force and os.path.exists(dst) and os.path.getsize(dst) > 0:
        print(f"Cached locally: {dst} ({os.path.getsize(dst):,} B)")
        return
    if not force:
        cached = find_cached_on_drive(os.path.basename(dst))
        if cached is not None:
            print(f"Using Drive cache: {cached} -> {dst}")
            shutil.copy2(cached, dst)
            return
    print(f"Downloading {url} -> {dst}")
    req = urllib.request.Request(url, headers={"User-Agent": "webGrainy-followup/1.0"})
    with urllib.request.urlopen(req, timeout=180) as resp, open(dst, "wb") as f:
        total = int(resp.headers.get("Content-Length", 0))
        with tqdm(total=total, unit="B", unit_scale=True, desc=os.path.basename(dst)) as pbar:
            while True:
                chunk = resp.read(1024 * 1024)
                if not chunk:
                    break
                f.write(chunk)
                pbar.update(len(chunk))

def check_sha256(path: str, expected_sha256: str, expected_bytes: int) -> tuple[bool, str]:
    actual_bytes = os.path.getsize(path)
    if actual_bytes != expected_bytes:
        return False, f"size {actual_bytes} != {expected_bytes}"
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    actual_sha = h.hexdigest()
    if actual_sha != expected_sha256:
        return False, f"sha256 {actual_sha} != {expected_sha256}"
    return True, f"{actual_bytes:,} B"

# 1. IG2_data.zip (Zenodo 17866827) -> extract directly to /content/data/IG2_data (NOT /content/test_images!)
MANIFEST_CSV = "/content/webGrainy/audit/followup/heldout_manifest.csv"
with open(MANIFEST_CSV, newline="", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))

needs_extract = any(not os.path.exists(os.path.join(DATA_DIR, r["image_relpath"])) for r in manifest_rows)
if needs_extract:
    drive_ig2_zip = find_cached_on_drive("IG2_data.zip")
    ig2_zip = drive_ig2_zip if drive_ig2_zip else "/content/data/IG2_data.zip"
    if drive_ig2_zip:
        print(f"Using Drive IG2_data.zip: {drive_ig2_zip}")
    else:
        download_file("https://zenodo.org/records/17866827/files/IG2_data.zip?download=1", ig2_zip)
    with zipfile.ZipFile(ig2_zip, "r") as zf:
        zf.extractall(DATA_DIR)
    print("Extracted IG2_data.zip to", DATA_DIR)
else:
    print("IG2_data already extracted at", DATA_DIR)

# 2. Pinned Hugging Face ONNX weights (src/modules/modelCache.ts)
MODEL_REVISION = "7773328a544ac050b7820667e68cbfb02a2e954a"
HF_BASE = f"https://huggingface.co/inYourOwnBrowser/imagegrains-2.0-onnx/resolve/{MODEL_REVISION}"
PINNED_MODELS = {
    "ig2_unet_fp32.onnx": ("5e692c5593ac15e614b77602e1d4f8c2f84dd939f4386b1f7e5641dbcd62053c", 26489665),
    "ig2_cpsam_fp16.onnx": ("944a114a0cb1a3a215c11affefe7211caac6da2b74895c496d5a7b5a23b6230d", 617364567),
}
for fname, (sha, sz) in PINNED_MODELS.items():
    p = os.path.join(WEIGHTS_DIR, fname)
    download_file(f"{HF_BASE}/{fname}", p)
    ok, msg = check_sha256(p, sha, sz)
    if not ok:
        print(f"Cached {fname} failed pinned check ({msg}); re-downloading pinned revision from HF...")
        download_file(f"{HF_BASE}/{fname}", p, force=True)
        ok, msg = check_sha256(p, sha, sz)
        assert ok, f"{fname}: {msg}"
    print(f"Verified SHA-256 OK: {fname} ({msg})")

# 3. Upstream PyTorch weights (Zenodo 15728186)
download_file("https://zenodo.org/records/15728186/files/IG2_full_set.200525?download=1", os.path.join(WEIGHTS_DIR, "IG2_full_set.200525"))
download_file("https://zenodo.org/records/15728186/files/IG2_full_set_cp_SAM?download=1", os.path.join(WEIGHTS_DIR, "IG2_full_set_cp_SAM"))

# 4. K1 demo images & physical caliper data (dmair1989/imagegrains)
GH_RAW = "https://raw.githubusercontent.com/dmair1989/imagegrains/main/demo_data"
download_file(f"{GH_RAW}/K1_field_measurement.csv", os.path.join(K1_DIR, "K1_field_measurement.csv"))
download_file(f"{GH_RAW}/K1/K1_C2_385.jpg", os.path.join(K1_DIR, "K1_C2_385.jpg"))
download_file(f"{GH_RAW}/K1/K1_C3_0449.jpg", os.path.join(K1_DIR, "K1_C3_0449.jpg"))

# 5. Verify true <SITE>/<train|test>/<file> paths on disk
for r in manifest_rows:
    img_p = os.path.join(DATA_DIR, r["image_relpath"])
    msk_p = os.path.join(DATA_DIR, r["mask_relpath"])
    assert os.path.exists(img_p), f"Missing image: {img_p}"
    assert os.path.exists(msk_p), f"Missing mask: {msk_p}"
    r["image_path"] = img_p
    r["mask_path"] = msk_p

print("Manifest verified on disk:", len(manifest_rows), "rows | splits:", Counter(r["official_split"] for r in manifest_rows))

In [ ]:
# Cell 3: Inference helpers, TS runner bridge, morphometrics/PQ, and 3-image ORT-Web WASM parity check
import json, math, os, subprocess, tempfile
import cv2
import numpy as np
import onnxruntime as ort
from skimage.measure import regionprops

REPO_DIR = "/content/webGrainy"
TS_RUNNER = os.path.join(REPO_DIR, "audit/followup/ts_postprocess_runner.mjs")

def load_rgb(path: str) -> np.ndarray:
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is not None:
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    from PIL import Image
    return np.array(Image.open(path).convert("RGB"))

def load_gt_mask(path: str, hw: tuple[int, int]) -> np.ndarray:
    if path.endswith(".npy"):
        arr = np.load(path)
    else:
        arr = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if arr is None:
            from PIL import Image
            arr = np.array(Image.open(path))
    if arr.ndim == 3:
        if np.array_equal(arr[:, :, 0], arr[:, :, 1]) and np.array_equal(arr[:, :, 0], arr[:, :, 2]):
            arr = arr[:, :, 0]
        else:
            rgb = arr[:, :, :3].astype(np.int64)
            arr = (rgb[:, :, 0] << 16) | (rgb[:, :, 1] << 8) | rgb[:, :, 2]
    if arr.shape[:2] != hw:
        from PIL import Image
        arr = np.array(Image.fromarray(arr.astype(np.int32), mode="I").resize((hw[1], hw[0]), Image.Resampling.NEAREST))
    arr_i = arr.astype(np.int64)
    arr_i[arr_i < 0] = 0
    uniq, inv = np.unique(arr_i, return_inverse=True)
    out = inv.reshape(hw).astype(np.int32)
    if len(uniq) > 0 and uniq[0] != 0:
        out += 1
    return out

def normalize_gray_unet(img_rgb: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    p1, p99 = np.percentile(gray, [1, 99])
    return ((gray - p1) / (p99 - p1 + 1e-8)).astype(np.float32)

def normalize_rgb_cpsam(img_rgb: np.ndarray) -> np.ndarray:
    arr = img_rgb.astype(np.float32)
    p1, p99 = np.percentile(arr, [1, 99])
    return ((arr - p1) / (p99 - p1 + 1e-8)).astype(np.float32)

def run_onnx_unet(sess: ort.InferenceSession, img_rgb: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    norm_gray = normalize_gray_unet(img_rgb)
    H, W = norm_gray.shape
    pad_h = (16 - (H % 16)) % 16
    pad_w = (16 - (W % 16)) % 16
    padded = norm_gray if (pad_h == 0 and pad_w == 0) else np.pad(norm_gray, ((0, pad_h), (0, pad_w)), mode="reflect")
    inp = np.zeros((1, 2, padded.shape[0], padded.shape[1]), dtype=np.float32)
    inp[0, 0] = padded
    out = sess.run(["flows_cellprob"], {"image": inp})[0][0][:, :H, :W]
    return out[:2].astype(np.float32), out[2].astype(np.float32)

def _tile_starts(length: int, tile_size: int = 256, stride: int = 192) -> list[int]:
    if length <= tile_size:
        return [0]
    starts = list(range(0, length - tile_size, stride))
    if not starts or starts[-1] != length - tile_size:
        starts.append(length - tile_size)
    return starts

def run_onnx_cpsam(sess: ort.InferenceSession, img_rgb: np.ndarray, tile_size: int = 256, overlap: int = 64, batch_size: int = 16) -> tuple[np.ndarray, np.ndarray]:
    norm_rgb = normalize_rgb_cpsam(img_rgb)
    H, W, _ = norm_rgb.shape
    pad_h = max(0, tile_size - H)
    pad_w = max(0, tile_size - W)
    if pad_h > 0 or pad_w > 0:
        norm_rgb = np.pad(norm_rgb, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
    Hp, Wp, _ = norm_rgb.shape
    stride = tile_size - overlap
    ys = _tile_starts(Hp, tile_size, stride)
    xs = _tile_starts(Wp, tile_size, stride)
    ramp = np.clip(np.hanning(tile_size).astype(np.float32), 1e-3, 1.0)
    win = np.outer(ramp, ramp)

    acc = np.zeros((3, Hp, Wp), dtype=np.float32)
    wsum = np.zeros((Hp, Wp), dtype=np.float32)
    coords = [(y, x) for y in ys for x in xs]
    for i in range(0, len(coords), batch_size):
        batch_coords = coords[i:i + batch_size]
        batch = np.stack([np.transpose(norm_rgb[y:y+tile_size, x:x+tile_size], (2, 0, 1)) for y, x in batch_coords], axis=0).astype(np.float16)
        preds = sess.run(["flows_cellprob"], {"image": batch})[0].astype(np.float32)
        for (y, x), pred in zip(batch_coords, preds):
            acc[:, y:y+tile_size, x:x+tile_size] += pred * win
            wsum[y:y+tile_size, x:x+tile_size] += win
    out = acc / np.maximum(wsum, 1e-6)
    return out[:2, :H, :W].astype(np.float32), out[2, :H, :W].astype(np.float32)

def run_ts_postprocess(dP: np.ndarray, cellprob: np.ndarray, max_area_frac: float) -> tuple[np.ndarray, list[dict]]:
    H, W = cellprob.shape
    with tempfile.TemporaryDirectory() as td:
        dp_p = os.path.join(td, "dp.f32")
        cp_p = os.path.join(td, "cp.f32")
        meta_p = os.path.join(td, "meta.json")
        out_json = os.path.join(td, "out.json")
        out_mask = os.path.join(td, "mask.i32")
        np.ascontiguousarray(dP, dtype="<f4").tofile(dp_p)
        np.ascontiguousarray(cellprob, dtype="<f4").tofile(cp_p)
        with open(meta_p, "w") as f:
            json.dump({
                "dPPath": dp_p,
                "cellprobPath": cp_p,
                "width": int(W),
                "height": int(H),
                "maxGrainAreaFraction": float(max_area_frac),
                "outputJsonPath": out_json,
                "outputMaskPath": out_mask,
            }, f)
        subprocess.run(["node", TS_RUNNER, "--input", meta_p], cwd=REPO_DIR, check=True)
        with open(out_json) as f:
            res = json.load(f)
        masks = np.fromfile(out_mask, dtype="<i4").reshape((H, W))
    return masks, res["grains"]

def compute_mask_grain_metrics(masks: np.ndarray, exclude_border_margin: int = 2) -> list[dict]:
    # Compute bAxisPx, dEqPx, areaPx for GT or PyTorch masks (interior grains).
    H, W = masks.shape
    grains = []
    for prop in regionprops(masks):
        area = float(prop.area)
        if area <= 0:
            continue
        minr, minc, maxr, maxc = prop.bbox
        is_border = (minr < exclude_border_margin or minc < exclude_border_margin or
                     maxr > H - exclude_border_margin or maxc > W - exclude_border_margin)
        if is_border:
            continue
        b_axis = float(prop.axis_minor_length) if hasattr(prop, "axis_minor_length") else float(prop.minor_axis_length)
        d_eq = math.sqrt(4.0 * area / math.pi)
        if b_axis > 0 and d_eq > 0:
            grains.append({"areaPx": area, "bAxisPx": b_axis, "dEqPx": d_eq})
    return grains

def area_weighted_percentiles(grains: list[dict], key: str, pcts=(16.0, 50.0, 84.0)) -> dict[str, float]:
    valid = [(float(g[key]), float(g["areaPx"])) for g in grains if not g.get("isBorder", False) and g.get(key, 0) > 0 and g.get("areaPx", 0) > 0]
    if not valid:
        return {f"D{int(p)}": float("nan") for p in pcts}
    valid.sort(key=lambda x: x[0])
    vals = np.array([v[0] for v in valid], dtype=np.float64)
    wts = np.array([v[1] for v in valid], dtype=np.float64)
    cum = np.cumsum(wts)
    # Midpoint CDF matching sedimentStats.ts
    cdf = (cum - 0.5 * wts) / cum[-1] * 100.0
    out = {}
    for p in pcts:
        out[f"D{int(p)}"] = float(np.interp(p, cdf, vals))
    return out

def compute_pq(pred: np.ndarray, gt: np.ndarray, iou_thresh: float = 0.5) -> float:
    pred_ids = np.unique(pred); pred_ids = pred_ids[pred_ids > 0]
    gt_ids = np.unique(gt); gt_ids = gt_ids[gt_ids > 0]
    if len(pred_ids) == 0 and len(gt_ids) == 0:
        return 1.0
    if len(pred_ids) == 0 or len(gt_ids) == 0:
        return 0.0
    p_areas = {int(i): int((pred == i).sum()) for i in pred_ids}
    g_areas = {int(j): int((gt == j).sum()) for j in gt_ids}
    # Fast pair overlap via 64-bit encoding
    mask = (pred > 0) & (gt > 0)
    if not np.any(mask):
        return 0.0
    pairs = pred[mask].astype(np.int64) * 1_000_000 + gt[mask].astype(np.int64)
    uniq_pairs, counts = np.unique(pairs, return_counts=True)
    tp = 0
    iou_sum = 0.0
    matched_p, matched_g = set(), set()
    candidates = []
    for code, inter in zip(uniq_pairs, counts):
        pi = int(code // 1_000_000)
        gi = int(code % 1_000_000)
        iou = float(inter) / float(p_areas[pi] + g_areas[gi] - int(inter))
        if iou > iou_thresh:
            candidates.append((iou, pi, gi))
    candidates.sort(reverse=True)
    for iou, pi, gi in candidates:
        if pi not in matched_p and gi not in matched_g:
            matched_p.add(pi)
            matched_g.add(gi)
            tp += 1
            iou_sum += iou
    fp = len(pred_ids) - tp
    fn = len(gt_ids) - tp
    denom = tp + 0.5 * fp + 0.5 * fn
    return float(iou_sum / denom) if denom > 0 else 0.0

# ORT-Web WASM vs Python ONNXRuntime parity sanity check on 3 test images (U-Net)
test_rows = [r for r in manifest_rows if r["official_split"] == "test"]
unet_sess = ort.InferenceSession(os.path.join(WEIGHTS_DIR, "ig2_unet_fp32.onnx"), providers=["CUDAExecutionProvider", "CPUExecutionProvider"])

wasm_check_script = r'''
import fs from "node:fs";
import path from "node:path";
import { pathToFileURL } from "node:url";

const ortDist = path.join(process.cwd(), "node_modules/onnxruntime-web/dist");
const ort = await import(pathToFileURL(path.join(ortDist, "ort.min.mjs")).href);
const meta = JSON.parse(fs.readFileSync(process.argv[2], "utf8"));
ort.env.wasm.numThreads = 1;
ort.env.wasm.wasmPaths = ortDist + "/";
const modelBytes = new Uint8Array(fs.readFileSync(meta.modelPath));
const session = await ort.InferenceSession.create(modelBytes, { executionProviders: ["wasm"] });
const buf = fs.readFileSync(meta.inputPath);
const f32 = new Float32Array(buf.buffer, buf.byteOffset, buf.byteLength / 4);
const tensor = new ort.Tensor("float32", f32, meta.shape);
const out = await session.run({ image: tensor });
const res = out.flows_cellprob.data;
fs.writeFileSync(meta.outputPath, Buffer.from(res.buffer, res.byteOffset, res.byteLength));
'''

wasm_diffs = []
with tempfile.TemporaryDirectory() as td:
    js_p = os.path.join(td, "wasm_check.mjs")
    with open(js_p, "w") as f:
        f.write(wasm_check_script)
    for r in test_rows[:3]:
        img_rgb = load_rgb(r["image_path"])
        ng = normalize_gray_unet(img_rgb)
        H, W = ng.shape
        ph, pw = (16 - H % 16) % 16, (16 - W % 16) % 16
        pad = ng if (ph == 0 and pw == 0) else np.pad(ng, ((0, ph), (0, pw)), mode="reflect")
        inp = np.zeros((1, 2, pad.shape[0], pad.shape[1]), dtype=np.float32)
        inp[0, 0] = pad
        py_out = unet_sess.run(["flows_cellprob"], {"image": inp})[0]
        inp_p = os.path.join(td, "inp.f32")
        out_p = os.path.join(td, "out.f32")
        meta_p = os.path.join(td, "meta.json")
        np.ascontiguousarray(inp, dtype="<f4").tofile(inp_p)
        with open(meta_p, "w") as f:
            json.dump({
                "modelPath": os.path.join(WEIGHTS_DIR, "ig2_unet_fp32.onnx"),
                "inputPath": inp_p,
                "outputPath": out_p,
                "shape": list(inp.shape),
            }, f)
        subprocess.run(["node", js_p, meta_p], cwd=REPO_DIR, check=True)
        wasm_out = np.fromfile(out_p, dtype="<f4").reshape(py_out.shape)
        max_abs = float(np.max(np.abs(py_out - wasm_out)))
        wasm_diffs.append({"uid": r["uid"], "max_abs_diff": max_abs})

print("ORT-Web WASM vs Python ORT (3 test images, U-Net):", wasm_diffs)

In [ ]:
# Cell 4: Run held-out evaluation (official_split == 'test', n=40) across PyTorch (venv_cp3/cp4), TS 0.15, and TS 1.0
import json, os, subprocess
import pandas as pd
from tqdm.auto import tqdm

# Toggle: "test_40" (default, official_split == 'test') or "all_243"
EVAL_SCOPE = "test_40"

if EVAL_SCOPE == "test_40":
    eval_rows = [r for r in manifest_rows if r["official_split"] == "test"]
elif EVAL_SCOPE == "all_243":
    eval_rows = list(manifest_rows)
else:
    raise ValueError(f"Unknown EVAL_SCOPE: {EVAL_SCOPE}")

print(f"Evaluating {len(eval_rows)} images (EVAL_SCOPE={EVAL_SCOPE})...")

PT_CACHE_DIR = os.path.join(OUT_DIR, "pt_cache")
os.makedirs(os.path.join(PT_CACHE_DIR, "unet"), exist_ok=True)
os.makedirs(os.path.join(PT_CACHE_DIR, "cpsam"), exist_ok=True)

pt_worker_code = """
import argparse, json, os
import cv2
import numpy as np
import torch
from PIL import Image
from cellpose import models

def load_rgb(path):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is not None:
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return np.array(Image.open(path).convert("RGB"))

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["unet", "cpsam"], required=True)
    ap.add_argument("--weights", required=True)
    ap.add_argument("--jobs", required=True)
    ap.add_argument("--out-dir", required=True)
    args = ap.parse_args()

    with open(args.jobs) as f:
        jobs = json.load(f)
    pending = [j for j in jobs if not os.path.exists(os.path.join(args.out_dir, f"{j['uid']}.npy"))]
    if not pending:
        print(f"[{args.model}] All {len(jobs)} PyTorch masks already cached.")
        return

    use_gpu = torch.cuda.is_available()
    cp_model = models.CellposeModel(gpu=use_gpu, pretrained_model=args.weights)
    for idx, j in enumerate(pending, 1):
        img_rgb = load_rgb(j["image_path"])
        if args.model == "unet":
            masks = cp_model.eval(
                img_rgb, channels=[0, 0], diameter=30.0,
                flow_threshold=0.4, cellprob_threshold=0.0, min_size=30, normalize=True
            )[0].astype(np.int32)
        else:
            masks = cp_model.eval(
                img_rgb, diameter=None,
                flow_threshold=0.4, cellprob_threshold=0.0, min_size=30, tile_overlap=0.25, normalize=True
            )[0].astype(np.int32)
        np.save(os.path.join(args.out_dir, f"{j['uid']}.npy"), masks)
        if idx % 10 == 0 or idx == len(pending):
            print(f"[{args.model}] PyTorch reference {idx}/{len(pending)} done")

if __name__ == "__main__":
    main()
"""

pt_worker_path = "/tmp/pt_eval_worker.py"
pt_jobs_path = "/tmp/pt_eval_jobs.json"
with open(pt_worker_path, "w") as f:
    f.write(pt_worker_code)
with open(pt_jobs_path, "w") as f:
    json.dump([{"uid": r["uid"], "image_path": r["image_path"]} for r in eval_rows], f)

print("Running upstream PyTorch U-Net (venv_cp3)...")
subprocess.run([
    "/content/venv_cp3/bin/python", pt_worker_path,
    "--model", "unet",
    "--weights", os.path.join(WEIGHTS_DIR, "IG2_full_set.200525"),
    "--jobs", pt_jobs_path,
    "--out-dir", os.path.join(PT_CACHE_DIR, "unet"),
], check=True)

print("Running upstream PyTorch CP-SAM (venv_cp4)...")
subprocess.run([
    "/content/venv_cp4/bin/python", pt_worker_path,
    "--model", "cpsam",
    "--weights", os.path.join(WEIGHTS_DIR, "IG2_full_set_cp_SAM"),
    "--jobs", pt_jobs_path,
    "--out-dir", os.path.join(PT_CACHE_DIR, "cpsam"),
], check=True)

cpsam_sess = ort.InferenceSession(os.path.join(WEIGHTS_DIR, "ig2_cpsam_fp16.onnx"), providers=["CUDAExecutionProvider", "CPUExecutionProvider"])

records = []
for r in tqdm(eval_rows, desc="ONNX + TS Engine"):
    img_rgb = load_rgb(r["image_path"])
    H, W = img_rgb.shape[:2]
    gt_masks = load_gt_mask(r["mask_path"], (H, W))
    gt_grains = compute_mask_grain_metrics(gt_masks)
    gt_b = area_weighted_percentiles(gt_grains, "bAxisPx")
    gt_d = area_weighted_percentiles(gt_grains, "dEqPx")

    for model_name in ["unet", "cpsam"]:
        pt_masks = np.load(os.path.join(PT_CACHE_DIR, model_name, f"{r['uid']}.npy")).astype(np.int32)
        if model_name == "unet":
            dP, cp = run_onnx_unet(unet_sess, img_rgb)
        else:
            dP, cp = run_onnx_cpsam(cpsam_sess, img_rgb)

        # PyTorch reference
        pt_grains = compute_mask_grain_metrics(pt_masks)
        pt_b = area_weighted_percentiles(pt_grains, "bAxisPx")
        pt_d = area_weighted_percentiles(pt_grains, "dEqPx")
        records.append({
            "uid": r["uid"], "site": r["site"], "official_split": r["official_split"],
            "is_k1_overlap": r["is_k1_overlap"], "model": model_name, "engine": "pytorch_ref",
            "n_gt": len(gt_grains), "n_pred": len(pt_grains),
            "gt_b_D16": gt_b["D16"], "gt_b_D50": gt_b["D50"], "gt_b_D84": gt_b["D84"],
            "pred_b_D16": pt_b["D16"], "pred_b_D50": pt_b["D50"], "pred_b_D84": pt_b["D84"],
            "pred_d_D50": pt_d["D50"], "pred_f_D50": float("nan"),
            "pq_vs_gt": compute_pq(pt_masks, gt_masks), "pq_vs_pt": 1.0,
        })

        # Shipped TS Engine: guard ON (0.15) and guard OFF (1.0)
        for guard_val, eng_name in [(0.15, "ts_v41_guard015"), (1.0, "ts_v41_guard100")]:
            ts_masks, ts_grains = run_ts_postprocess(dP, cp, guard_val)
            ts_int = [g for g in ts_grains if not g.get("touchesBorder", g.get("isBorder", False))]
            ts_b = area_weighted_percentiles(ts_int, "bAxisPx")
            ts_d = area_weighted_percentiles(ts_int, "dEqPx")
            ts_f = area_weighted_percentiles(ts_int, "minFeretPx")
            records.append({
                "uid": r["uid"], "site": r["site"], "official_split": r["official_split"],
                "is_k1_overlap": r["is_k1_overlap"], "model": model_name, "engine": eng_name,
                "n_gt": len(gt_grains), "n_pred": len(ts_int),
                "gt_b_D16": gt_b["D16"], "gt_b_D50": gt_b["D50"], "gt_b_D84": gt_b["D84"],
                "pred_b_D16": ts_b["D16"], "pred_b_D50": ts_b["D50"], "pred_b_D84": ts_b["D84"],
                "pred_d_D50": ts_d["D50"], "pred_f_D50": ts_f["D50"],
                "pq_vs_gt": compute_pq(ts_masks, gt_masks), "pq_vs_pt": compute_pq(ts_masks, pt_masks),
            })

df_res = pd.DataFrame(records)
per_img_csv = os.path.join(OUT_DIR, "heldout_results_per_image.csv")
df_res.to_csv(per_img_csv, index=False)
print("Saved:", per_img_csv, "rows:", len(df_res))

In [ ]:
# Cell 5: Summary statistics with bootstrap 95% CIs, metric shifts, and K1 physical GT comparison
import json
import numpy as np
import pandas as pd

def bootstrap_ci(arr: np.ndarray, fn, n_boot: int = 2000, seed: int = 42) -> list[float]:
    rng = np.random.default_rng(seed)
    if len(arr) < 2:
        val = float(fn(arr)) if len(arr) == 1 else float("nan")
        return [val, val]
    boots = np.empty(n_boot, dtype=np.float64)
    n = len(arr)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[i] = fn(arr[idx])
    return [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))]

def summarize_col(sub: pd.DataFrame, gt_col: str, pred_col: str) -> dict:
    valid = sub[(sub["n_gt"] > 0) & sub[gt_col].notna() & sub[pred_col].notna() & (sub[gt_col] > 0)]
    gt = valid[gt_col].to_numpy(dtype=np.float64)
    pr = valid[pred_col].to_numpy(dtype=np.float64)
    abs_pct = 100.0 * np.abs(pr - gt) / gt
    signed_pct = 100.0 * (pr - gt) / gt
    return {
        "n": int(len(valid)),
        "mean_ape_pct": float(np.mean(abs_pct)),
        "mean_ape_ci95": bootstrap_ci(abs_pct, np.mean),
        "median_ape_pct": float(np.median(abs_pct)),
        "median_ape_ci95": bootstrap_ci(abs_pct, np.median),
        "p90_ape_pct": float(np.percentile(abs_pct, 90)),
        "frac_gt25_pct": float(100.0 * np.mean(abs_pct > 25.0)),
        "frac_gt50_pct": float(100.0 * np.mean(abs_pct > 50.0)),
        "median_signed_bias_pct": float(np.median(signed_pct)),
        "mean_signed_bias_pct": float(np.mean(signed_pct)),
    }

summary = {"eval_scope": EVAL_SCOPE, "wasm_parity_3img": wasm_diffs, "models": {}, "metric_shifts_d50": {}, "k1_physical": {}}

for model_name in ["unet", "cpsam"]:
    summary["models"][model_name] = {}
    for eng_name in ["pytorch_ref", "ts_v41_guard015", "ts_v41_guard100"]:
        sub = df_res[(df_res["model"] == model_name) & (df_res["engine"] == eng_name)]
        summary["models"][model_name][eng_name] = {
            "D16": summarize_col(sub, "gt_b_D16", "pred_b_D16"),
            "D50": summarize_col(sub, "gt_b_D50", "pred_b_D50"),
            "D84": summarize_col(sub, "gt_b_D84", "pred_b_D84"),
            "pq_vs_gt_mean": float(sub["pq_vs_gt"].mean()),
            "pq_vs_pt_mean": float(sub["pq_vs_pt"].mean()),
        }

# Area-weighted D50 shifts between bAxis, dEq, and minFeret (Workstream C)
for model_name in ["unet", "cpsam"]:
    sub = df_res[(df_res["model"] == model_name) & (df_res["engine"] == "ts_v41_guard015")]
    valid = sub[sub["pred_b_D50"].notna() & sub["pred_d_D50"].notna() & sub["pred_f_D50"].notna() & (sub["pred_b_D50"] > 0)]
    b = valid["pred_b_D50"].to_numpy()
    d = valid["pred_d_D50"].to_numpy()
    f = valid["pred_f_D50"].to_numpy()
    summary["metric_shifts_d50"][model_name] = {
        "n": int(len(valid)),
        "dEq_vs_bAxis_pct_median": float(np.median(100.0 * (d - b) / b)),
        "minFeret_vs_bAxis_pct_median": float(np.median(100.0 * (f - b) / b)),
        "minFeret_vs_dEq_pct_median": float(np.median(100.0 * (f - d) / d)),
    }

# K1 Physical GT comparison (0.39 mm/px, field CSV in cm -> mm)
k1_df = pd.read_csv(os.path.join(K1_DIR, "K1_field_measurement.csv"))
k1_cands = [c for c in k1_df.columns if "b" in c.lower() or "axis" in c.lower()]
if not k1_cands:
    k1_cands = list(k1_df.select_dtypes(include="number").columns) or [k1_df.columns[-1]]
k1_col = k1_cands[0]
k1_vals = pd.to_numeric(k1_df[k1_col], errors="coerce").dropna().to_numpy(dtype=np.float64)
# K1_field_measurement.csv is in cm (1.3-22.5 cm -> 13-225 mm); convert to mm if median < 50
k1_mm = k1_vals * 10.0 if ("cm" in k1_col.lower() or np.median(k1_vals) < 50.0) else k1_vals
summary["k1_physical"]["field_n"] = int(len(k1_mm))
summary["k1_physical"]["field_D16_mm"] = float(np.percentile(k1_mm, 16))
summary["k1_physical"]["field_D50_mm"] = float(np.percentile(k1_mm, 50))
summary["k1_physical"]["field_D84_mm"] = float(np.percentile(k1_mm, 84))
summary["k1_physical"]["caveat"] = "K1_C2_385 and K1_C3_0449 overlap with APF_2/train and APF_2/test in IG2_data.zip."

MM_PER_PX = 0.39
for model_name in ["unet", "cpsam"]:
    pooled_grains = []
    for k1_name in ["K1_C2_385.jpg", "K1_C3_0449.jpg"]:
        img_rgb = load_rgb(os.path.join(K1_DIR, k1_name))
        dP, cp = run_onnx_unet(unet_sess, img_rgb) if model_name == "unet" else run_onnx_cpsam(cpsam_sess, img_rgb)
        _, grains = run_ts_postprocess(dP, cp, 0.15)
        pooled_grains.extend([g for g in grains if not g.get("touchesBorder", g.get("isBorder", False))])
    summary["k1_physical"][model_name] = {
        "n_grains_pooled": int(len(pooled_grains)),
        "bAxis_mm": {k: v * MM_PER_PX for k, v in area_weighted_percentiles(pooled_grains, "bAxisPx").items()},
        "dEq_mm": {k: v * MM_PER_PX for k, v in area_weighted_percentiles(pooled_grains, "dEqPx").items()},
        "minFeret_mm": {k: v * MM_PER_PX for k, v in area_weighted_percentiles(pooled_grains, "minFeretPx").items()},
    }

summary_path = os.path.join(OUT_DIR, "heldout_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print("Saved:", summary_path)
if DRIVE_MOUNTED:
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    for fn in ["heldout_results_per_image.csv", "heldout_summary.json"]:
        src_p = os.path.join(OUT_DIR, fn)
        if os.path.exists(src_p):
            shutil.copy2(src_p, os.path.join(DRIVE_ROOT, fn))
    print("Copied outputs to Drive:", DRIVE_ROOT)
print(json.dumps(summary, indent=2))